# 01 · Preparación de datos

Este notebook construye, a partir de los CSV de PyRadiomics, el dataset que usan los notebooks **02 · Modelo y métricas** y **03 · Evaluación de rigor**.

**Entrada:** un CSV por imagen de resonancia, extraído con PyRadiomics sobre la segmentación de FreeSurfer, con una fila por región cerebral (`LabelID`) y todas sus métricas.

**Salida:** una fila por imagen con las características de **forma** de las regiones comunes a todas las imágenes, normalizadas por el volumen total segmentado, y la tabla demográfica.

**Cohortes** (todas de ADNI):

| Target | Cohorte | Significado |
|---|---|---|
| 0 | Ctrl | Pacientes con DCL estable (sMCI) durante más de 4 años. Se usa la imagen de la primera visita. |
| 1 | 1Y | Imagen adquirida **12 meses antes** del diagnóstico de enfermedad de Alzheimer (EA). |
| 2 | 2Y | Imagen adquirida **24 meses antes** del diagnóstico de EA. |

Muchos sujetos aportan una imagen a 1Y y otra a 2Y (la misma persona, con un año de diferencia), así que cada imagen se asocia a su **sujeto biológico** (`Sujeto_Real`) para que las particiones de los notebooks siguientes nunca repartan a una persona entre entrenamiento y prueba.

**Archivos que se generan:**
- `datos_procesados/` (dataset procesado, demografía y volúmenes crudos). Son datos derivados de ADNI: **no se suben a git**.
- `Tabla_Control_Calidad.xlsx`: contiene identificadores de ADNI, tampoco se sube.
- `Tabla_Demografica.xlsx`.
- Las figuras se guardan en `img/` (PNG a 300 dpi), además de mostrarse aquí.

## 0. Configuración

Rutas de entrada y salida, y el umbral de prevalencia que define las regiones anatómicas de referencia en el control de calidad (sección 2.1).

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from ml_core import (NOMBRES_COHORTE, TEXTURE_FAMILIES, cargar_cohorte, purge_texture_features, normalize_by_proxy_etiv,
                     load_region_names, plot_cohort_distribution, plot_region_prevalence, plot_invalid_volumes,
                     plot_feature_families, plot_total_volume, plot_volume_vs_total, plot_pca_comparison, plot_demographics)

# Carpetas con un CSV de PyRadiomics por imagen (Target -> carpeta)
RUTAS_COHORTES = {1: './subjects/ADNI_1Y/', 2: './subjects/ADNI_2Y/', 0: './subjects/MCI_Constant_FreeSurfer/'}

# Excel con los pares de imágenes 1Y/2Y del mismo sujeto (hoja 2: '1Y image_id', '2Y image_id', 'PTID')
RUTA_EXCEL = './patients_1_to_2copia.xlsx'

# Colecciones de imágenes exportadas de ADNI/IDA, una por cohorte (sujeto, sexo, edad, visita y fecha)
RUTA_METADATOS = './adni_meta'
ARCHIVOS_IDA = {1: 'MCI_to_AD_1Y.csv', 2: 'MCI_to_AD_2Y.csv', 0: 'ConstantMCI.csv'}

# Tabla de nombres de regiones de FreeSurfer (LabelID -> nombre anatómico); se traducen al español en ml_core/regions.py
RUTA_LUT = './FreeSurferColorLUT.csv'

# Salida: datos derivados de ADNI, ignorados por git
DIR_SALIDA = './datos_procesados'
os.makedirs(DIR_SALIDA, exist_ok=True)

PREVALENCIA_MINIMA = 0.99
NOMBRES_REGION, NOMBRES_REGION_ES = load_region_names(RUTA_LUT)

## 1. Carga de las cohortes

Cada CSV es una imagen en formato largo: una fila por región segmentada, con más de 100 métricas de PyRadiomics (forma, primer orden y textura). `cargar_cohorte` lee todos los CSV de una carpeta y les asigna su `Target`.

In [ ]:
df_largo = pd.concat([cargar_cohorte(ruta, target) for target, ruta in RUTAS_COHORTES.items()], ignore_index=True)

imagenes_por_cohorte = df_largo.groupby('Target')['PatientID'].nunique().rename(index=NOMBRES_COHORTE)
print(f"{len(df_largo)} filas (imagen × región) y {df_largo.shape[1]} columnas")
imagenes_cargadas = imagenes_por_cohorte.to_dict()
print(f"Imágenes por cohorte: {imagenes_cargadas}")
print(f"Regiones por imagen: entre {df_largo.groupby('PatientID')['LabelID'].nunique().min()} "
      f"y {df_largo.groupby('PatientID')['LabelID'].nunique().max()}")

## 2. Control de calidad de las segmentaciones

Antes de construir el dataset se excluyen las imágenes cuya segmentación no es válida. Se aplican **criterios explícitos**, no identificadores fijos, y cada exclusión queda registrada con su motivo en `exclusiones`, que al final se exporta como `Tabla_Control_Calidad.xlsx`.

El control de calidad se aplica **antes** de la intersección de regiones (sección 3). Si no, una sola imagen incompleta eliminaría esas regiones de todas las demás.

In [ ]:
exclusiones = []

def registrar_exclusion(patient_id, target, criterio, detalle):
    exclusiones.append({'PatientID': patient_id, 'Cohorte': NOMBRES_COHORTE[target],
                        'Criterio': criterio, 'Detalle': detalle})

### 2.1 Regiones anatómicas faltantes

FreeSurfer segmenta las mismas estructuras anatómicas en todos los cerebros. Por eso, si a una imagen le falta una región que está en al menos el 99 % de las imágenes, su segmentación está incompleta (por ejemplo, porque la adquisición no cubrió parte del cerebro).

Las regiones con menor prevalencia (vasos, 5.º ventrículo, hipointensidades de sustancia blanca, regiones "undetermined") no son constantes anatómicas: aparecen o no según el cerebro. No se usan como criterio de exclusión; la intersección de la sección 3 las descarta.

La gráfica de barras muestra el porcentaje de imágenes en que aparece cada región que no está en el 100 %: en azul las que se exigen (presentes en al menos el 99 %) y en gris las opcionales.

In [ ]:
n_imagenes = df_largo['PatientID'].nunique()
prevalencia = df_largo.groupby('LabelID')['PatientID'].nunique() / n_imagenes
regiones_referencia = set(prevalencia[prevalencia >= PREVALENCIA_MINIMA].index)
print(f"Regiones presentes en ≥ {PREVALENCIA_MINIMA:.0%} de las imágenes: {len(regiones_referencia)}")

# Regiones que no están en todas las imágenes: las de referencia (≥ 99 %) y las de presencia variable
plot_region_prevalence(prevalencia, NOMBRES_REGION_ES, PREVALENCIA_MINIMA)

# Imágenes a las que les falta alguna región de referencia
regiones_por_imagen = df_largo.groupby(['PatientID', 'Target'])['LabelID'].agg(set)
for (patient_id, target), regiones in regiones_por_imagen.items():
    faltantes = sorted(regiones_referencia - regiones)
    if faltantes:
        nombres = ', '.join(NOMBRES_REGION_ES.get(l, str(l)) for l in faltantes)
        registrar_exclusion(patient_id, target, 'Regiones anatómicas faltantes', f"{len(faltantes)} regiones: {nombres}")
        print(f"{patient_id} ({NOMBRES_COHORTE[target]}): faltan {len(faltantes)} regiones ({nombres})")

### 2.2 Volúmenes o áreas no positivos

En una segmentación válida, el volumen (`MeshVolume`, `VoxelVolume`), el área (`SurfaceArea`) y la razón superficie/volumen de cada región son estrictamente positivos. Un valor ≤ 0 indica una malla de superficie corrupta: todas las métricas de forma de esa región dejan de tener sentido.

La figura compara, en las regiones afectadas, el volumen de la imagen corrupta (puntos rojos) con la mediana de las demás imágenes (gris).

In [ ]:
metricas_positivas = ['shape_MeshVolume', 'shape_VoxelVolume', 'shape_SurfaceArea', 'shape_SurfaceVolumeRatio']
columnas_positivas = [c for c in df_largo.columns if any(c.endswith(m) for m in metricas_positivas)]

filas_invalidas = df_largo[(df_largo[columnas_positivas] <= 0).any(axis=1)]
for i, ((patient_id, target), regiones) in enumerate(filas_invalidas.groupby(['PatientID', 'Target'])):
    labels = sorted(regiones['LabelID'].unique().tolist())
    registrar_exclusion(patient_id, target, 'Volumen o área no positivos', f"{len(labels)} regiones: {labels}")
    print(f"{patient_id} ({NOMBRES_COHORTE[target]}): {len(labels)} regiones con valores ≤ 0")
    display(regiones[['LabelID'] + columnas_positivas].head())
    plot_invalid_volumes(df_largo, patient_id, labels, NOMBRES_REGION_ES, filename=f"QC_Volumenes_No_Positivos_{i + 1}")

# Se retiran las imágenes que no pasaron el control de calidad
excluidas_qc = {e['PatientID'] for e in exclusiones}
df_largo = df_largo[~df_largo['PatientID'].isin(excluidas_qc)]
print(f"Imágenes excluidas por control de calidad: {len(excluidas_qc)}. Quedan {df_largo['PatientID'].nunique()}.")

## 3. Intersección de regiones y aplanado

Para que todas las imágenes tengan exactamente las mismas columnas, se conservan solo las regiones presentes en **todas** las imágenes que pasaron el control de calidad.

Después se aplana el formato largo (una fila por imagen y región) a formato ancho: **una fila por imagen**, con una columna por combinación región × métrica (`Label_<LabelID>_<métrica>`).

In [ ]:
conteo_regiones = df_largo.groupby('LabelID')['PatientID'].nunique()
labels_validos = sorted(conteo_regiones[conteo_regiones == df_largo['PatientID'].nunique()].index)
df_largo = df_largo[df_largo['LabelID'].isin(labels_validos)]
print(f"Regiones comunes a todas las imágenes: {len(labels_validos)}")

df_melted = df_largo.melt(id_vars=['PatientID', 'LabelID', 'Target'], var_name='Caracteristica', value_name='Valor')
df_melted['Columna'] = 'Label_' + df_melted['LabelID'].astype(str) + '_' + df_melted['Caracteristica']
df_ancho = df_melted.pivot_table(index=['PatientID', 'Target'], columns='Columna', values='Valor', aggfunc='first').reset_index()
df_ancho.columns.name = None
print(f"Dataset aplanado: {df_ancho.shape[0]} filas × {df_ancho.shape[1] - 2} características")

## 4. Imágenes repetidas entre cohortes

Un mismo archivo de imagen no puede estar a la vez a 12 y a 24 meses del diagnóstico, ni ser de un conversor y de un DCL estable. Si una imagen aparece en dos cohortes, se conserva en una sola, con este orden de prioridad:
- **1Y antes que 2Y:** se conserva el horizonte más cercano al diagnóstico.
- **Conversor antes que Ctrl:** la conversión observada prevalece sobre la etiqueta de estabilidad.

La asignación se verifica después con la visita registrada en IDA (sección 5).

In [ ]:
PRIORIDAD_COHORTE = {1: 0, 2: 1, 0: 2}

df_ancho['_prioridad'] = df_ancho['Target'].map(PRIORIDAD_COHORTE)
df_ancho = df_ancho.sort_values(['PatientID', '_prioridad'])
repetidas = df_ancho.duplicated(subset=['PatientID'], keep='first')

for patient_id, target in zip(df_ancho.loc[repetidas, 'PatientID'], df_ancho.loc[repetidas, 'Target']):
    conservada = df_ancho.loc[(df_ancho['PatientID'] == patient_id) & ~repetidas, 'Target'].iloc[0]
    registrar_exclusion(patient_id, target, 'Imagen repetida en dos cohortes', f"Se conserva solo en {NOMBRES_COHORTE[conservada]}")
    print(f"{patient_id}: aparece en {NOMBRES_COHORTE[target]} y {NOMBRES_COHORTE[conservada]}; se conserva en {NOMBRES_COHORTE[conservada]}")

df_maestro = df_ancho[~repetidas].drop(columns='_prioridad').sort_values(['PatientID', 'Target']).reset_index(drop=True)

## 5. Sujeto biológico de cada imagen (metadatos de IDA)

Los nombres de archivo son identificadores de imagen: `PI<Image Data ID>` en los conversores y el PTID en los controles. Para agrupar por persona hace falta el **PTID** de cada imagen. La fuente es la columna `Subject` de las colecciones exportadas de IDA, que además aportan el sexo, la edad al escaneo, la visita y la fecha de adquisición (sección 10).

El mapeo se coteja con el Excel de pares 1Y/2Y; cualquier discrepancia detiene la ejecución.

In [ ]:
# Metadatos de IDA indexados con el mismo PatientID que los CSV de radiómica
frames = []
for target, archivo in ARCHIVOS_IDA.items():
    ida = pd.read_csv(os.path.join(RUTA_METADATOS, archivo))
    patient_id = ida['Subject'] if target == 0 else 'PI' + ida['Image Data ID'].str[1:]
    frames.append(pd.DataFrame({'PatientID': patient_id, 'Target': target, 'Subject': ida['Subject'],
                                'Sexo': ida['Sex'], 'Edad': ida['Age'], 'Visita': ida['Visit'],
                                'Fecha_Adquisicion': pd.to_datetime(ida['Acq Date'], format='%m/%d/%Y')}))
metadatos = pd.concat(frames, ignore_index=True)

# Toda imagen del dataset debe tener registro en IDA
sin_registro = sorted(set(df_maestro['PatientID']) - set(metadatos['PatientID']))
assert not sin_registro, f"Imágenes sin registro en IDA: {sin_registro[:5]}"
df_maestro['Sujeto_Real'] = df_maestro['PatientID'].map(metadatos.drop_duplicates('PatientID').set_index('PatientID')['Subject'])

# Verificación contra el Excel de pares 1Y/2Y
pares = pd.read_excel(RUTA_EXCEL, sheet_name=1)
mapeo_excel = {}
for columna in ['1Y image_id', '2Y image_id']:
    for image_id, ptid in zip(pares[columna], pares['PTID']):
        if pd.notna(image_id):
            mapeo_excel[f"PI{int(image_id)}"] = ptid
mapeo_ida = df_maestro.set_index('PatientID')['Sujeto_Real']
cotejadas = [pid for pid in mapeo_excel if pid in mapeo_ida.index]
discrepancias = {pid: (mapeo_excel[pid], mapeo_ida[pid]) for pid in cotejadas if mapeo_excel[pid] != mapeo_ida[pid]}
assert not discrepancias, f"El Excel y IDA asignan distinto sujeto a: {discrepancias}"
print(f"{df_maestro['Sujeto_Real'].nunique()} sujetos en {len(df_maestro)} imágenes. "
      f"{len(cotejadas)} imágenes cotejadas contra el Excel, sin discrepancias.")

# Verificación de la imagen repetida de la sección 4 con su visita en IDA
repetidas_ida = metadatos[metadatos['PatientID'].isin([e['PatientID'] for e in exclusiones if e['Criterio'] == 'Imagen repetida en dos cohortes'])]
display(repetidas_ida[['PatientID', 'Target', 'Subject', 'Visita', 'Fecha_Adquisicion']])

Según el Excel de pares, el sujeto de la imagen repetida recibió el diagnóstico de EA en la visita m48. Una imagen de la visita m36 queda a 12 meses del diagnóstico, así que pertenece a **1Y**, como se asignó en la sección 4.

## 6. Controles que convirtieron a EA

La cohorte Ctrl se define como DCL **estable** durante más de 4 años. Si el sujeto de un control también aparece en una cohorte de conversión (1Y o 2Y), convirtió a EA en algún momento del seguimiento y no es un sMCI. Su imagen de control se excluye y se conserva la de conversor.

Sin esta corrección, la misma persona podría quedar con etiquetas opuestas y repartida entre entrenamiento y prueba.

In [ ]:
sujetos_conversores = set(df_maestro.loc[df_maestro['Target'] != 0, 'Sujeto_Real'])
conflicto = (df_maestro['Target'] == 0) & df_maestro['Sujeto_Real'].isin(sujetos_conversores)

for _, fila in df_maestro[conflicto].iterrows():
    otras = df_maestro[(df_maestro['Sujeto_Real'] == fila['Sujeto_Real']) & (df_maestro['Target'] != 0)]
    detalle = ', '.join(f"{NOMBRES_COHORTE[t]} ({p})" for p, t in zip(otras['PatientID'], otras['Target']))
    registrar_exclusion(fila['PatientID'], 0, 'Sujeto también en cohorte de conversión', f"Convirtió a EA durante el seguimiento: {detalle}")

df_maestro = df_maestro[~conflicto].reset_index(drop=True)
print(f"Controles excluidos por haber convertido: {conflicto.sum()}")

## 7. Reporte de exclusiones y tamaño final de las cohortes

Resumen de todas las exclusiones, con su criterio. Se exporta para citarlo en la metodología de la tesis; contiene identificadores de ADNI, así que no se sube a git.

La gráfica muestra cuántas imágenes conserva cada cohorte y qué proporción del total representa.

In [ ]:
reporte_exclusiones = pd.DataFrame(exclusiones)
reporte_exclusiones.to_excel('Tabla_Control_Calidad.xlsx', index=False)
display(reporte_exclusiones)

imagenes = df_maestro['Target'].value_counts().sort_index().rename(index=NOMBRES_COHORTE)
print(f"Imágenes finales: {imagenes.to_dict()} (total {len(df_maestro)})")
compartidos = df_maestro.groupby('Sujeto_Real')['Target'].agg(set).apply(lambda t: {1, 2} <= t).sum()
print(f"Sujetos con imagen en 1Y y en 2Y: {compartidos}")

plot_cohort_distribution(imagenes.to_dict())

## 8. Preprocesamiento de las características

### 8.1 Valores no numéricos, infinitos y columnas constantes

Se convierten todas las métricas a número, los infinitos a faltantes y los faltantes a la mediana de su columna. Se espera 0 faltantes; se reporta el conteo para comprobarlo. Las columnas constantes (varianza ≤ 1e-5) no aportan información y se eliminan.

Ningún paso usa la etiqueta, así que no hay fuga de información. Aun así, el pipeline de los notebooks 02 y 03 repite el filtro de varianza dentro de cada fold.

In [ ]:
columnas_meta = ['PatientID', 'Target', 'Sujeto_Real']
caracteristicas = df_maestro.drop(columns=columnas_meta).apply(pd.to_numeric, errors='coerce')
caracteristicas = caracteristicas.replace([np.inf, -np.inf], np.nan)
print(f"Valores faltantes o infinitos: {int(caracteristicas.isna().sum().sum())}")
caracteristicas = caracteristicas.fillna(caracteristicas.median())

varianza = caracteristicas.var()
caracteristicas = caracteristicas[varianza[varianza > 1e-5].index]
print(f"Columnas constantes eliminadas: {(varianza <= 1e-5).sum()}")

df_limpio = pd.concat([df_maestro[columnas_meta], caracteristicas], axis=1)

# Copia con todas las familias y sin normalizar, para el PCA de la sección 9
df_todas = df_limpio.copy()

### 8.2 Solo características de forma

Las familias de primer orden y de textura (`firstorder`, `glcm`, `glrlm`, `glszm`, `gldm`, `ngtdm`) dependen de la intensidad de la imagen, es decir, del brillo y el contraste de cada escáner. Se eliminan y se conservan solo los descriptores de **forma** (familia `shape`): volumen, área, esfericidad, elongación, diámetros, etc.

La gráfica de barras muestra cuántos descriptores calcula cada familia por región; en azul, la que se conserva.

In [ ]:
# Descriptores por región de cada familia de PyRadiomics (antes de la purga)
descriptores_region = {c.split('_original_')[1] for c in df_maestro.columns if '_original_' in c}
descriptores_por_familia = pd.Series({familia: sum(d.startswith(familia + '_') for d in descriptores_region)
                                      for familia in ['shape'] + TEXTURE_FAMILIES})
plot_feature_families(descriptores_por_familia)

df_limpio = purge_texture_features(df_limpio)
columnas_forma = [c for c in df_limpio.columns if c not in columnas_meta]
descriptores = sorted({c.split('shape_')[1] for c in columnas_forma})
print(f"Características de forma: {len(columnas_forma)} ({len(descriptores)} descriptores × {len(labels_validos)} regiones, "
      f"menos las constantes)")
print("Descriptores:", ', '.join(descriptores))

### 8.3 Normalización por volumen total segmentado ("Proxy eTIV")

Los volúmenes y áreas absolutos dependen del tamaño de la cabeza. Para compararlos entre personas, cada `MeshVolume`, `VoxelVolume` y `SurfaceArea` regional se divide entre la **suma de esa métrica en las regiones segmentadas** de la misma imagen, y pasa a ser una proporción del total.

Este total **no es el eTIV** (volumen intracraneal estimado) de FreeSurfer: es el volumen segmentado, de ~1.1 L frente a ~1.5 L de un eTIV real. Además incluye la atrofia global, así que difiere por cohorte (se muestra abajo). Las longitudes, diámetros y descriptores adimensionales no se normalizan.

Se guardan tres figuras en `img/`:
- `Volumen_Total_Segmentado`: el denominador, por cohorte.
- `Normalizacion_Hipocampo_Antes` y `Normalizacion_Hipocampo_Despues`: el hipocampo izquierdo frente al volumen total, antes y después de normalizar. Si la normalización funciona, la relación con el tamaño de la cabeza (la pendiente de la nube de puntos) desaparece.

In [ ]:
# Volumen total segmentado por cohorte (antes de normalizar), en litros
total_voxel = df_limpio[[c for c in columnas_forma if c.endswith('shape_VoxelVolume')]].sum(axis=1) / 1e6
display(total_voxel.groupby(df_limpio['Target']).agg(['mean', 'std']).rename(index=NOMBRES_COHORTE).round(3))

HIPOCAMPO_IZQ = 'Label_17_original_shape_VoxelVolume'
hipocampo_crudo = df_limpio[HIPOCAMPO_IZQ].copy()
df_limpio = normalize_by_proxy_etiv(df_limpio)

# Tres figuras: se guardan sin mostrarse
cohortes = df_limpio['Target'].map(NOMBRES_COHORTE).to_numpy()
hipocampo_norm = 100 * df_limpio[HIPOCAMPO_IZQ]
rutas = [plot_total_volume(total_voxel, cohortes, show=False),
         plot_volume_vs_total(total_voxel, hipocampo_crudo, cohortes, f"{NOMBRES_REGION_ES[17]}: volumen (mm³)",
                              'Normalizacion_Hipocampo_Antes', show=False),
         plot_volume_vs_total(total_voxel, hipocampo_norm, cohortes, f"{NOMBRES_REGION_ES[17]}: % del volumen total",
                              'Normalizacion_Hipocampo_Despues', show=False)]
print("Figuras guardadas:", *rutas, sep='\n')

# Dependencia del tamaño de la cabeza: correlación con el volumen total, antes y después de normalizar
print(f"Correlación del {NOMBRES_REGION_ES[17].lower()} con el volumen total: "
      f"antes r = {np.corrcoef(total_voxel, hipocampo_crudo)[0, 1]:.2f}, "
      f"después r = {np.corrcoef(total_voxel, hipocampo_norm)[0, 1]:.2f}")

# Verificación: sin valores faltantes ni infinitos tras la normalización
valores = df_limpio[columnas_forma].to_numpy(dtype=float)
assert np.isfinite(valores).all(), "Hay valores faltantes o infinitos tras la normalización"
print(f"Dataset procesado: {df_limpio.shape[0]} imágenes × {len(columnas_forma)} características, sin faltantes ni infinitos.")

## 9. Exploración: PCA antes y después de la purga

Proyección de todas las imágenes en las dos primeras componentes principales, en dos versiones lado a lado:
- **Izquierda, todas las familias de PyRadiomics** (forma, primer orden y textura) **sin normalizar**: la copia `df_todas` de la sección 8.1, escalada con `StandardScaler` y sin recortar los ejes. Es el mismo procedimiento del PCA con el que se detectó el efecto de lote entre FastSurfer y FreeSurfer (antes y después de ComBat, en la tesis).
- **Derecha, solo las características de forma normalizadas:** las que usan los modelos, escaladas con `RobustScaler`. Los ejes ignoran el 1 % de valores extremos.

Con todas las familias aparece un grupo formado sobre todo por Ctrl que sigue a la fase de adquisición de ADNI, es decir, al escáner y al protocolo, no al software de segmentación. Las componentes que lo separan están dominadas por características de intensidad y textura. Con solo las características de forma, la distribución es más homogénea.

In [ ]:
columnas_todas = [c for c in df_todas.columns if c not in columnas_meta]
plot_pca_comparison(df_todas[columnas_todas], df_limpio[columnas_forma], df_limpio['Target'], NOMBRES_REGION_ES)

## 10. Demografía por cohorte

Edad y sexo al momento del escaneo, de los metadatos de IDA. El periodo de adquisición sirve como aproximación al tipo de escáner: las imágenes anteriores a septiembre de 2009 son del periodo ADNI1, adquirido mayoritariamente a 1.5 T.

Cada cohorte de conversión se compara con Ctrl:
- Variables continuas: Mann-Whitney U.
- Variables binarias: χ², o Fisher si alguna celda tiene menos de 5.

1Y vs 2Y no se prueba, porque comparten sujetos y las muestras no son independientes. Las diferencias de edad se corrigen en los notebooks 02 y 03 ajustando cada característica por edad y sexo dentro de cada fold.

In [ ]:
INICIO_ADNIGO = pd.Timestamp('2009-09-01')

demografia = df_limpio[['PatientID', 'Target', 'Sujeto_Real']].merge(
    metadatos[['PatientID', 'Target', 'Sexo', 'Edad', 'Visita', 'Fecha_Adquisicion']],
    on=['PatientID', 'Target'], how='left', validate='one_to_one')
demografia.index = df_limpio.index
assert demografia['Edad'].notna().all()
demografia['Sexo_F'] = (demografia['Sexo'] == 'F').astype(int)
demografia['Anio_Adquisicion'] = (demografia['Fecha_Adquisicion'].dt.year
                                  + (demografia['Fecha_Adquisicion'].dt.dayofyear - 1) / 365.25)
demografia['Periodo_ADNI1'] = (demografia['Fecha_Adquisicion'] < INICIO_ADNIGO).astype(int)

In [ ]:
def media_de(x):
    return f"{x.mean():.1f} ± {x.std():.1f}"

def mediana_ric(x):
    return f"{x.median():.1f} [{x.quantile(0.25):.1f}–{x.quantile(0.75):.1f}]"

def conteo(x):
    return f"{int(x.sum())} ({100 * x.mean():.1f}%)"

def p_continua(a, b):
    return stats.mannwhitneyu(a, b).pvalue

def p_binaria(a, b):
    tabla = [[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]]
    if min(min(fila) for fila in tabla) < 5:
        return stats.fisher_exact(tabla).pvalue
    return stats.chi2_contingency(tabla).pvalue

grupos = {nombre: demografia[demografia['Target'] == t] for t, nombre in NOMBRES_COHORTE.items()}
variables = [
    ('Edad al escaneo (años), media ± DE', 'Edad', media_de, p_continua, 'Mann-Whitney U'),
    ('Edad al escaneo (años), mediana [RIC]', 'Edad', mediana_ric, None, ''),
    ('Mujeres, n (%)', 'Sexo_F', conteo, p_binaria, 'χ² / Fisher'),
    ('Año de adquisición, mediana [RIC]', 'Anio_Adquisicion', mediana_ric, p_continua, 'Mann-Whitney U'),
    ('Adquirida en periodo ADNI1 (< sep-2009), n (%)', 'Periodo_ADNI1', conteo, p_binaria, 'χ² / Fisher'),
]

filas = [{'Variable': 'Imágenes, n', **{g: str(len(d)) for g, d in grupos.items()}},
         {'Variable': 'Sujetos únicos, n', **{g: str(d['Sujeto_Real'].nunique()) for g, d in grupos.items()}}]
for etiqueta, columna, resumen, prueba, nombre_prueba in variables:
    fila = {'Variable': etiqueta, **{g: resumen(d[columna]) for g, d in grupos.items()}}
    if prueba is not None:
        fila['p (1Y vs Ctrl)'] = prueba(grupos['1Y'][columna], grupos['Ctrl'][columna])
        fila['p (2Y vs Ctrl)'] = prueba(grupos['2Y'][columna], grupos['Ctrl'][columna])
        fila['Prueba'] = nombre_prueba
    filas.append(fila)

tabla_demografica = pd.DataFrame(filas).set_index('Variable')
tabla_demografica.to_excel('Tabla_Demografica.xlsx')
display(tabla_demografica)

Edad media (la línea negra es ± 1 desviación estándar), proporción de mujeres y proporción de imágenes del periodo ADNI1 en cada cohorte. Los valores p están en la tabla anterior.

In [ ]:
plot_demographics(demografia)

## 11. Guardar los datos procesados

Se guardan en `datos_procesados/`, que está ignorada por git:
- `dataset_procesado.csv`: la entrada de los modelos (identificadores + características de forma normalizadas).
- `demografia.csv`: edad, sexo y fecha por imagen. Se usan como covariables de ajuste, nunca como características.
- `forma_cruda.csv`: características de forma **sin normalizar**, para el análisis longitudinal pareado del notebook 03.

También se exportan `ctrl_vs_1Y.csv` y `ctrl_vs_2Y.csv` con los subconjuntos de cada comparativa contra Ctrl, que también están ignorados por git.

In [ ]:
df_limpio.to_csv(os.path.join(DIR_SALIDA, 'dataset_procesado.csv'), index=False)
demografia.to_csv(os.path.join(DIR_SALIDA, 'demografia.csv'), index=False)
purge_texture_features(df_maestro).to_csv(os.path.join(DIR_SALIDA, 'forma_cruda.csv'), index=False)

df_limpio[df_limpio['Target'].isin([0, 1])].to_csv('ctrl_vs_1Y.csv', index=False)
df_limpio[df_limpio['Target'].isin([0, 2])].to_csv('ctrl_vs_2Y.csv', index=False)
print(f"Guardado en {DIR_SALIDA}: {sorted(os.listdir(DIR_SALIDA))}")